In [1]:
import numpy as np
import math

The key to recognize here is because the angles subtended by $T$ in each circle form $60^\circ$, the angle at $T$ must be $120^\circ$ for each section. That is $\triangle TBC$, $\triangle TCA$ and $\triangle TAB$ are all ones with the largest angle as $120^\circ$. With that, we can apply the law of cosines on each triangle to show
$$
a^2 = q^2 + r^2 - 2qr \cos(120^\circ) \implies a^2 = q^2 + r^2 + qr \\
b^2 = p^2 + q^2 - 2pq \cos(120^\circ) \implies b^2 = p^2 + q^2 + pq \\
c^2 = p^2 + r^2 - 2pr \cos(120^\circ) \implies c^2 = p^2 + r^2 + pr
$$

This tells us we need to generate triples $(p, q, r)$ such that any pair of those $(x,y)$ has $x^2 + y^2 + xy$ being a perfect square. The goal is then to generate all such pairs, and then map them on with the property on $(p, q, r)$.

To generate the pairs, we think of Euclid's formula to generate Pythagorean triples where $x^2 + y^2$ is a perfect square. Can we do something similar here? Suppose the perfect square is $d^2$. Then we need to find triples $(x, y, d)$ such that $x^2 + y^2 + xy = d^2$. Dividing through by $d^2$ and using the substitution $w = \frac{x}{d}$ and $z = \frac{y}{d}$, we get $w^2 + z^2 + wz = 1$. Graphing this on the $w,z$-plane gives an ellipse. It is obvious that the point $w = 0, z = -1$ passes through the ellipse. The parametrization of all lines through $(w, z) = (0, -1)$ is $(w, z) = (w, tw - 1)$ where $1 \leq t$. Then we find the intersection of such a line with the ellipse to get
$$
\begin{align*}
w^2 + z^2 + wz = 1 \implies & w^2 + (tw - 1)^2 + w(tw - 1) = 1 \\
    \implies & w^2 + t^2w^2 - 2tw + 1 + tw^2 - w = 1 \\
    \implies & (1 + t + t^2) w^2 - (2t + 1) w = 0 \\
    \implies & (1 + t + t^2) w = 2t + 1 \\
    \implies & w = \frac{2t+1}{1 + t + t^2} \\
    \implies & z = \frac{t^2 - 1}{1 + t + t^2}
\end{align*}
$$

Because of the substitution here, if $x, y, d$ are all positive integers, then $w, z$ must be positive rational numbers. For these to be rational, we need $t$ to be rational. Suppose $t = \frac{m}{n}$. Then
$$
w = \frac{2\left(\frac{m}{n}\right) + 1}{1 + \frac{m}{n} + \left(\frac{m}{n}\right)^2} = \frac{2mn + n^2}{n^2 + mn + m^2} \\
z = \frac{\left(\frac{m}{n}\right)^2 - 1}{1 + \frac{m}{n} + \left(\frac{m}{n}\right)^2} = \frac{m^2 - n^2}{n^2 + mn + m^2}
$$

Thus we can simply get the transformation if we find generate with integers $m,n$ such that $x = 2mn + n^2$ and $y = m^2 - n^2$ (which makes $d = n^2 + mn + m^2$). Now to generate these uniquely (i.e., primitives) we may think to just apply the same rule as Euclid's formula that $m > n$ and $\gcd(m,n) = 1$. However, if we stopped there we would be missing one condition. In particular, if $m \equiv n \bmod 3$, then
$$
x = 2mn + n^2 = 3mn - mn + n^2 = 3mn - (m - n)n \; | \; 3 \\
y = m^2 - n^2 = (m+n)(m-n) \; | \; 3 \\
d = m^2 + mn + n^2 = m^2 - 2mn + n^2 + 3mn = (m-n)^2 + 3mn \; | \; 3
$$

Thus we also need to add the condition that $m \not\equiv n \bmod 3$. Then the algorithm works as follows:
1. For every $m, n \in \mathbb{N}$ such that $m > n$, $\gcd(m,n) = 1$, and $m \not\equiv n \bmod 3$, we generate $x = 2mn + n^2$ and $y = m^2 - n^2$, which generates a pair such that $x^2 + y^2 + xy$ is a perfect square. Then we generate all multiples $(kx, ky)$ since $(kx)^2 + (ky)^2 + (kx)(ky) = k^2 x^2 + k^2 y^2 + k^2 xy = k^2 (x^2 + y^2 + xy)$ is also a perfect square. To keep everything in order, reassing $x$ and $y$ so that $x := \min(x,y)$ and $y := \max(x,y)$.
2. Loop through every pair generated from (1) to form a dictionary $vps$ that maps the every $x$ as a key to each of the $y$ that go with it. Every element of $vps[x]$ is greater than $x$ but still satisfies the perfect square condition from (1).
3. For every $p$ that is a key of $vps$, loop through its $qs = vps[p]$. For each $q$ in $qs$, see if $vps[q]$ has anything. If it does, let $rs = vps[p] \;\cap\; vps[q]$. Then any elements of $rs$ are solutions such that $p, q, r$ are the segments in the problem.
4. For every solution $p, q, r$, save only unique sums.

In [2]:
def gcd(a,b):
    if b == 0:
        return a
    return gcd(b, a%b)

# generate pairs where x^2 + xy + y^2 is a perfect square
def pairs(limit):
    pairs = []
    
    for m in range(1, limit):
        for n in range(1, m):
            # require m > n
            if gcd(m,n) > 1 or (m-n)%3 == 0: continue

            # x = 2mn + n^2, y = m^2 - n^2
            a, b = 2*m*n + n*n, m*m - n*n

            if a+b >= limit: break

            a,b = min([a,b]), max([a,b])
            pairs.append((a,b))

            k = 2
            while sum(pairs[-1]) < limit:
                pairs.append((k*a, k*b))
                k += 1

    return pairs


In [9]:
limit = 120000 + 1
# generate all pairs that could work as side lengths
valid_pairs = pairs(limit)

# generate dictionary that maps shorter side to all longer sides
vps = {}
for p, q in valid_pairs:
    vps[p] = vps.get(p, set())
    vps[p].add(q)

sums_seen = set()

In [10]:
# for every shorter segment p
for p in vps:
    # check all its longer segments
    qs = vps[p]
    for q in qs:
        # if vps[p] intersects vps[q], then there are solutions in rs
        rs = vps[p].intersection(vps.get(q, set()))
        for r in rs:
            # if p + q + r < limit, then we get the solution
            if p + q + r < limit:
                print(p,q,r)
                sums_seen.add(p + q + r)

195 264 325
264 325 440
360 1015 3864
384 805 1520
390 528 650
435 1656 4669
528 650 880
585 792 975
720 2030 7728
765 1064 5016
768 1610 3040
780 1056 1300
792 975 1320
870 3312 9338
885 9499 12600
975 1320 1625
1029 15680 87720
1056 1300 1760
1080 3045 11592
1152 2415 4560
1170 1584 1950
1272 2065 4928
1305 4968 14007
1320 1625 2200
1365 1848 2275
1365 5472 6435
1440 4060 15456
1530 2128 10032
1536 3220 6080
1560 2112 2600
1560 9315 24552
1584 1950 2640
1740 6624 18676
1755 2376 2925
1770 18998 25200
1785 8415 11704
1800 5075 19320
1848 2275 3080
1920 4025 7600
1950 2640 3250
2064 15561 65520
2112 2600 3520
2145 2904 3575
2160 6090 23184
2175 8280 23345
2295 3192 15048
2304 4830 9120
2340 3168 3900
2376 2925 3960
2409 42735 58400
2451 10320 77805
2520 7105 27048
2535 3432 4225
2544 4130 9856
2610 9936 28014
2640 3250 4400
2655 28497 37800
2688 5635 10640
2709 4515 110960
2730 10944 12870
2730 3696 4550
2880 8120 30912
2904 3575 4840
2925 3960 4875
3045 11592 32683
3060 4256 20064
307

In [11]:
sum(sums_seen)

30758397